# 複数の分類器を交差検証で比較する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/09_classification/compare_classifiers.ipynb)

NLP クックブック 09-04 ／ テキスト分類

ナイーブベイズ、ロジスティック回帰、SVM などのうち、どれが自分のデータに合うかを公平に比べたい。

In [1]:
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

cats = ["rec.sport.baseball", "sci.space", "comp.graphics", "talk.politics.mideast", "rec.autos", "sci.med"]
data = fetch_20newsgroups(subset="train", categories=cats, remove=("headers", "footers", "quotes"))

models = {
    "MultinomialNB": MultinomialNB(alpha=0.1),
    "ComplementNB": ComplementNB(alpha=0.1),
    "LogisticRegression": LogisticRegression(C=10, max_iter=1000),
    "LinearSVC": LinearSVC(C=1),
    "SGD (hinge)": SGDClassifier(random_state=0),
    "RandomForest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
rows = []
for name, model in models.items():
    pipe = make_pipeline(TfidfVectorizer(stop_words="english", sublinear_tf=True, min_df=2), model)
    res = cross_validate(pipe, data.data, data.target, cv=cv, scoring=["accuracy", "f1_macro"])
    rows.append({"model": name, "accuracy": res["test_accuracy"].mean(), "±": res["test_accuracy"].std(),
                 "macro-F1": res["test_f1_macro"].mean(), "学習時間 (秒)": res["fit_time"].mean()})
pd.DataFrame(rows).round(3).sort_values("macro-F1", ascending=False)

,model,accuracy,±,macro-F1,学習時間 (秒)
1,ComplementNB,0.896,0.007,0.897,0.269
0,MultinomialNB,0.890,0.010,0.891,0.229
3,LinearSVC,0.879,0.012,0.880,0.270
2,LogisticRegression,0.878,0.011,0.879,18.999
4,SGD (hinge),0.872,0.015,0.872,0.340
5,RandomForest,0.818,0.008,0.819,4.047


- ベクトル化を `Pipeline` に入れておくのが重要。そうしないと、IDF の計算に検証データが混ざって評価が甘くなる (リーク)。
- 高次元で疎なテキスト特徴には線形モデル (LR、線形 SVM、NB) が強く、決定木系は相性が良くない。